# 09 | Exploratory crop-group mapping in Koumbia, Burkina Faso
**Author:** Ibrahim Kabore | **Mapping year:** 2022 | **Grid:** 10 m, WGS 84 / UTM zone 30N

The frozen Notebook 08C model uses six contextualised monthly Presto vectors (May–October, 768 features) and Histogram Gradient Boosting. Its 2022 retrospective accuracy was 53.05%, balanced accuracy 49.36%, and macro F1 47.63%. It overpredicts cereals and is strongly overconfident. This map is an exploratory spatial inference demonstration, not an operational crop inventory or an independent area estimate.

We request a small 3 × 3 km pilot cube around the median training location. A separate global WorldCereal cropland head supplies a provisional mask. Only the three trained crop groups can be predicted; other crops may be assigned incorrectly. Raster codes 0 and 255 distinguish masked-out non-cropland from missing or insufficient data.

The workflow uses the exact WorldCereal revision and encoder from Notebook 08C. The monthly preprocessing API is documented in the [pinned source](https://github.com/WorldCereal/worldcereal-classification/blob/bf8667902f583ec71fcca6b1a7bd1e48f64000f7/src/worldcereal/openeo/preprocessing.py).

## 1. Environment and frozen classifier
Select a GPU runtime (T4 or L4). Run the installation cell and mount Drive. Verify the manifest, class order, feature order and file hash before loading the classifier.

In [ ]:
import sys, subprocess
REVISION = "bf8667902f583ec71fcca6b1a7bd1e48f64000f7"
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
    f"worldcereal @ git+https://github.com/WorldCereal/worldcereal-classification.git@{REVISION}"])

In [ ]:
from google.colab import drive
from pathlib import Path
import json, hashlib, zipfile, io, requests
from importlib.metadata import version
drive.mount("/content/drive")
ROOT = Path("/content/drive/MyDrive/ESA_EO_Projects")
PROJECT = ROOT / "WorldCereal_Burkina_Faso"
DATA = PROJECT / "data"
MODELS = PROJECT / "models"
MODEL_DIR = MODELS / "crop_groups"
OUT = PROJECT / "outputs" / "mapping_2022"
OUT.mkdir(parents=True,exist_ok=True)
manifest = json.loads((MODEL_DIR/"classifier_manifest.json").read_text())
if version("scikit-learn") != manifest["sklearn_version"]:
    subprocess.check_call([sys.executable,"-m","pip","install","-q",f"scikit-learn=={manifest['sklearn_version']}"])
import numpy as np, pandas as pd, xarray as xr, torch, joblib, sklearn
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.patches import Patch
from tqdm.auto import tqdm
from pyproj import CRS, Transformer
import rasterio
from rasterio.transform import from_origin
from prometheo.models import Presto
from prometheo.models.pooling import PoolingMethods
from prometheo.predictors import Predictors, S1_BANDS, S2_BANDS, METEO_BANDS, DEM_BANDS, NODATAVALUE
from worldcereal.openeo.parameters import DEFAULT_SEASONAL_MODEL_URL
from worldcereal.utils.models import load_model_artifact
from worldcereal.local_inference import run_seasonal_inference
assert torch.cuda.is_available(), "Select a Colab GPU"
assert sklearn.__version__==manifest["sklearn_version"], "Restart runtime after installing scikit-learn"
assert version("prometheo")==manifest["prometheo"] and version("worldcereal")==manifest["worldcereal"]
assert manifest["worldcereal_revision"]==REVISION
assert manifest["configuration"]=="monthly" and manifest["model"]=="hgb"
POOL_MONTHS = [5,6,7,8,9,10]
FEATURES = [f"m{m:02d}_ft_{i}" for m in POOL_MONTHS for i in range(128)]
assert manifest["feature_names"]==FEATURES and manifest["feature_count"]==768
assert manifest["pool_months"]==POOL_MONTHS and manifest["context"]=="Jan-Dec" and manifest["use_radar"] is False and manifest["mask_b8a"] is True
model_path = MODEL_DIR/manifest["classifier_file"]
assert hashlib.sha256(model_path.read_bytes()).hexdigest()==manifest["classifier_sha256"]
classifier = joblib.load(model_path)
CLASSES = list(classifier.classes_)
assert CLASSES==manifest["classes"]==["Cereals","Cotton","Legumes"]
assert classifier.n_features_in_==768
assert manifest["crop_codes"]=={str(i+1):str(v) for i,v in enumerate(CLASSES)}
print("Frozen model verified:",CLASSES, "| features:",classifier.n_features_in_)
print("GPU:",torch.cuda.get_device_name(0))


In [ ]:
EXPECTED_SHA256 = "e16dce35b55b368c1553dd6c6dfe26f39d27818f24a9d710e2922cd4f00fed88"
MODEL_URL = DEFAULT_SEASONAL_MODEL_URL
assert "202608211601.zip" in MODEL_URL, MODEL_URL
french_bundle = ROOT / "WorldCereal_Crop_Mapping/models/2026/global_seasonal_20260821.zip"
bundle = french_bundle if french_bundle.is_file() else MODELS / "global_seasonal_20260821.zip"
if not bundle.is_file():
    r = requests.get(MODEL_URL, timeout=(30,120))
    r.raise_for_status()
    assert zipfile.is_zipfile(io.BytesIO(r.content))
    bundle.write_bytes(r.content)
artifact = load_model_artifact(bundle, cache_root=Path("/content/wc_cache"), encoder_only=True)
config = artifact.manifest
encoder_path = artifact.extract_dir / config["artifacts"]["checkpoints"]["encoder_only"]
fingerprint = hashlib.sha256(encoder_path.read_bytes()).hexdigest()
assert fingerprint == EXPECTED_SHA256, f"Unexpected encoder: {fingerprint}"
declared = (config.get("backbone") or {}).get("fingerprint")
if declared:
    assert fingerprint == declared
encoder = Presto()
encoder.load_state_dict(torch.load(encoder_path, map_location="cpu", weights_only=True), strict=True)
encoder = encoder.to("cuda").eval()
encoder.requires_grad_(False)
print("Verified:", fingerprint)
print("Checkpoint:", bundle)
assert fingerprint==manifest["checkpoint_sha256"]

## 2. Define a small pilot area
The default square is centred on the median projected coordinates of the training observations, rounded to the 10 m grid. This avoids choosing a location based on 2022 accuracy. Change SIDE_M only before submitting a new job. A pilot square is not the administrative boundary of Koumbia.

In [ ]:
MAP_YEAR = 2022
SIDE_M = 3000
EPSG = 32630
BATCH = 256
TILE = 40
BAND_MAP = {**{f"S2-L2A-{b}":b.replace("B0","B") for b in ["B02","B03","B04","B05","B06","B07","B08","B8A","B11","B12"]},
    "S1-SIGMA0-VV":"VV","S1-SIGMA0-VH":"VH","AGERA5-PRECIP":"precipitation",
    "AGERA5-TMEAN":"temperature","elevation":"elevation","slope":"slope"}
BANDS = list(BAND_MAP)
DATES = pd.date_range(f"{MAP_YEAR}-01-01",periods=12,freq="MS")
ref = pd.read_parquet(DATA/"burkina_extractions_clean.parquet",columns=["ref_id","sample_id","lon","lat","split"])
ref = ref.loc[ref.split.eq("train")].drop_duplicates(["ref_id","sample_id"])
assert len(ref)==992
project = Transformer.from_crs(4326,EPSG,always_xy=True)
inverse = Transformer.from_crs(EPSG,4326,always_xy=True)
xx,yy = project.transform(ref.lon.to_numpy(),ref.lat.to_numpy())
cx,cy = (int(round(np.median(q)/10)*10) for q in [xx,yy])
assert SIDE_M>0 and SIDE_M%20==0 and SIDE_M<=5000
west,south,east,north = cx-SIDE_M//2,cy-SIDE_M//2,cx+SIDE_M//2,cy+SIDE_M//2
AOI = {"west":west,"south":south,"east":east,"north":north,"epsg":EPSG}
RUN = OUT/f"pilot_{cx}_{cy}_{SIDE_M}m"
RUN.mkdir(parents=True,exist_ok=True)
print("AOI:",AOI)
print("Centre lon/lat:",inverse.transform(cx,cy))
print("Requested area km²:",SIDE_M**2/1e6)
fig,ax=plt.subplots(figsize=(7,5))
ax.scatter(xx,yy,s=4,alpha=.3,label="Training reference locations")
ax.add_patch(plt.Rectangle((west,south),SIDE_M,SIDE_M,fill=False,color="red",linewidth=2,label="Pilot area"))
ax.set_aspect("equal")
ax.legend()
ax.set(xlabel="Easting (m)",ylabel="Northing (m)",title="Pilot extent")
plt.show()


## 3. Request compatible monthly data from openEO
The official WorldCereal preprocessing function creates a 10 m Sentinel-2 cloud-masked monthly median cube plus Sentinel-1, AgERA5 and terrain layers. Radar is acquired for the separate global cropland head but masked for our crop classifier. Raw encoded units are retained. Authentication uses your own Copernicus Data Space account. This service consumes openEO credits; review your account before submission. Set SUBMIT_JOB=True deliberately to create a new job. The saved job ID allows resuming after a disconnect. If a previous job failed, inspect it before deliberately creating a replacement. Do not delete the job ID merely because processing is slow.

In [ ]:
import openeo
from openeo_gfmap import Backend, BackendContext, BoundingBoxExtent, TemporalContext
from worldcereal.openeo.preprocessing import worldcereal_preprocessed_inputs
from worldcereal.job import DEFAULT_INPUTS_JOB_OPTIONS
CUBE = RUN/"worldcereal_inputs_2022.nc"
JOB_RECORD = RUN/"openeo_job.json"
SUBMIT_JOB = False  # Change to True only when ready to spend openEO credits.
if not CUBE.is_file():
    if not JOB_RECORD.is_file() and not SUBMIT_JOB:
        raise RuntimeError("Review the AOI and credits, set SUBMIT_JOB=True, then rerun this cell.")
    conn = openeo.connect("openeo.dataspace.copernicus.eu").authenticate_oidc()
    if JOB_RECORD.is_file():
        state = json.loads(JOB_RECORD.read_text())
        assert state["aoi"]==AOI and state["year"]==MAP_YEAR and state["revision"]==REVISION
        job = conn.job(state["job_id"])
    else:
        bbox = BoundingBoxExtent(west,south,east,north,epsg=EPSG)
        temporal = TemporalContext(f"{MAP_YEAR}-01-01",f"{MAP_YEAR}-12-31")
        cube = worldcereal_preprocessed_inputs(connection=conn,backend_context=BackendContext(Backend.CDSE),
            spatial_extent=bbox,temporal_extent=temporal,target_epsg=EPSG,compositing_window="month",
            optical_mask_method="mask_scl_dilation",tile_size=128)
        cube = cube.filter_bbox(dict(bbox))
        options = dict(DEFAULT_INPUTS_JOB_OPTIONS)
        job = cube.save_result("NetCDF").create_job(title="Koumbia 2022 monthly inputs",job_options=options)
        JOB_RECORD.write_text(json.dumps({"job_id":job.job_id,"aoi":AOI,"year":MAP_YEAR,"revision":REVISION},indent=2))
    print("Job ID:",job.job_id)
    status = job.status()
    if status in ["error","canceled"]:
        raise RuntimeError(f"Job {job.job_id} status {status}; inspect job logs before retrying")
    if status!="finished":
        job.start_and_wait()
    assert job.status()=="finished"
    downloaded = RUN/"download"
    downloaded.mkdir(exist_ok=True)
    job.get_results().download_files(downloaded)
    candidates = sorted(downloaded.rglob("*.nc"))
    assert len(candidates)==1, f"Expected one NetCDF asset, got {candidates}"
    import shutil
    shutil.copyfile(candidates[0],CUBE)
print("Input cube:",CUBE)


## 4. Validate grid, dates and raw data
Reject an incompatible cube rather than silently substituting bands or rescaling. Terrain is static and may be broadcast across months. Weather and optical values retain the raw WorldCereal integer encoding. No artificial monthly observations are created.

In [ ]:
ds = xr.open_dataset(CUBE, mask_and_scale=True)
assert set(BANDS)<=set(ds.data_vars), sorted(set(BANDS)-set(ds.data_vars))
assert {"t","y","x"}<=set(ds.dims)
ds = ds.sortby("t").sortby("x").sortby("y",ascending=False)
assert pd.DatetimeIndex(ds.t.values).equals(DATES), ds.t.values
crs_data = ds["crs"].attrs
crs = CRS.from_wkt(crs_data.get("spatial_ref") or crs_data["crs_wkt"])
assert crs.to_epsg()==EPSG
assert np.allclose(np.diff(ds.x.values),10) and np.allclose(np.diff(ds.y.values),-10)
h,w = ds.sizes["y"],ds.sizes["x"]
assert h*w<=260000 and h>0 and w>0,(h,w)
assert abs(float(ds.x.min())-west)<30 and abs(float(ds.x.max())-east)<30
assert abs(float(ds.y.min())-south)<30 and abs(float(ds.y.max())-north)<30
transform = from_origin(float(ds.x[0])-5,float(ds.y[0])+5,10,10)
for b in BANDS:
    if "t" not in ds[b].dims:
        assert b in ["elevation","slope"], b
        ds[b] = ds[b].expand_dims(t=ds.t)
    assert set(ds[b].dims)=={"t","x","y"},(b,ds[b].dims)
    assert not ds[b].attrs.get("scale_factor"),f"Unexpected encoded scaling metadata: {b}"
    assert not ds[b].attrs.get("add_offset"),f"Unexpected offset: {b}"
    v = ds[b].isel(x=slice(None,None,5),y=slice(None,None,5)).values.astype("float32")
    v = v[np.isfinite(v)&(v!=65535)&(v!=NODATAVALUE)]
    print(b, "valid subsample:",len(v),"median:",round(float(np.median(v)),2) if len(v) else None)
    if b.startswith("S2") and len(v):
        assert 0<=np.min(v) and np.max(v)<=65534
    if b=="AGERA5-TMEAN":
        assert len(v)>0 and 20000<np.median(v)<40000, "Temperature must be Kelvin ×100"
    if b=="AGERA5-PRECIP":
        assert len(v)>0 and np.min(v)>=0
print("Cube:",h,w,"pixels |",crs)
ds = ds[BANDS+["crs"]]


## 5. Generate a provisional cropland mask
The global cropland head uses the same pretrained bundle, with all acquired modalities and its own annual prediction. This is not a locally validated cropland map. Mask values: 0 = predicted non-cropland, 1 = predicted cropland, 255 = unavailable. No crop groups are assigned outside predicted cropland.

In [ ]:
MASK_PATH = RUN/"cropland_mask_2022.tif"
def write_raster(path,arr,nodata,dtype,description):
    with rasterio.open(path,"w",driver="GTiff",height=h,width=w,count=1,dtype=dtype,
                       crs=crs.to_wkt(),transform=transform,nodata=nodata,compress="deflate",tiled=True,
                       blockxsize=128,blockysize=128) as dst:
        dst.write(arr.astype(dtype),1)
        dst.set_band_description(1,description)
        dst.update_tags(mapping_year=str(MAP_YEAR),purpose="exploratory research only")
if MASK_PATH.is_file():
    with rasterio.open(MASK_PATH) as r:
        assert r.shape==(h,w) and r.crs.to_epsg()==EPSG and r.transform==transform
        maskv = r.read(1)
else:
    mask_result = run_seasonal_inference(ds,seasonal_model_zip=bundle,season_ids=["tc-annual"],
        season_windows={"tc-annual":(f"{MAP_YEAR}-01-01",f"{MAP_YEAR}-12-31")},
        enable_cropland_head=True,enable_croptype_head=False,mask_cropland=False,
        batch_size=BATCH,device="cuda",mask_b8a=True,cache_root=Path("/content/wc_cache"),epsg=EPSG)
    mask = mask_result["cropland_classification"].transpose("y","x")
    assert np.array_equal(mask.x,ds.x) and np.array_equal(mask.y,ds.y)
    maskv = mask.values.astype(np.uint8)
    write_raster(MASK_PATH,maskv,255,"uint8","Global annual cropland prediction")
assert set(np.unique(maskv)).issubset({0,1,255})
assert (maskv==1).any(),"No cropland predicted in this pilot area"
print("Cropland-mask pixel counts:",dict(zip(*np.unique(maskv,return_counts=True))))
plt.imshow(np.ma.masked_equal(maskv,255),cmap="Greens",vmin=0,vmax=1)
plt.title("Provisional global cropland mask")
plt.axis("off")
plt.show()


## 6. Reproduce the frozen monthly encoder
The function below is identical to Notebook 08C. January–December inputs produce twelve contextualised embeddings; the May–October six vectors are flattened in month-major order (not averaged). Radar and B8A are masked for this classifier. Apply the same B08 coverage requirement used in training.

In [ ]:
def encode(values, lon, lat, dates, use_radar):
    n,t = len(lon),len(dates)
    assert t==12
    s1 = np.full((n,1,1,t,len(S1_BANDS)),NODATAVALUE,np.float32)
    s2 = np.full((n,1,1,t,len(S2_BANDS)),NODATAVALUE,np.float32)
    meteo = np.full((n,1,1,t,len(METEO_BANDS)),NODATAVALUE,np.float32)
    dem = np.full((n,1,1,len(DEM_BANDS)),NODATAVALUE,np.float32)

    for source,band in BAND_MAP.items():
        a = np.asarray(values[source],dtype=np.float32).copy()
        assert a.shape==(n,t)
        ok = np.isfinite(a) & (a!=NODATAVALUE) & (a!=65535)
        a[~ok] = NODATAVALUE
        if band in S2_BANDS:
            if band=="B8A":
                a[:] = NODATAVALUE
            s2[:,0,0,:,S2_BANDS.index(band)] = a
        elif band in S1_BANDS:
            if use_radar:
                ok &= a>0
                a[~ok] = NODATAVALUE
                a[ok] = 20*np.log10(a[ok])-83
                s1[:,0,0,:,S1_BANDS.index(band)] = a
        elif band in METEO_BANDS:
            a[ok] /= 100000 if band=="precipitation" else 100
            meteo[:,0,0,:,METEO_BANDS.index(band)] = a
        elif band in DEM_BANDS:
            dem[:,0,0,DEM_BANDS.index(band)] = a[:,0]
        else:
            raise ValueError(band)

    dates = pd.DatetimeIndex(dates)
    stamps = np.stack([dates.day,dates.month,dates.year],axis=1)
    predictor = Predictors(s1=s1,s2=s2,meteo=meteo,dem=dem,
        latlon=np.stack([lat,lon],axis=1).astype("float32").reshape(n,1,1,2),
        timestamps=np.broadcast_to(stamps,(n,t,3)).copy())
    with torch.inference_mode():
        z = encoder(predictor,eval_pooling=PoolingMethods.TIME).reshape(n,t,128)
        keep = np.flatnonzero(dates.month.isin(POOL_MONTHS)).tolist()
        return z[:,keep,:].cpu().numpy()

In [ ]:
def read_values(tile):
    out = {}
    for b in BANDS:
        arr = tile[b].transpose("y","x","t").values.astype(np.float32).reshape(-1,12)
        arr[(arr==65535)|(arr==NODATAVALUE)|~np.isfinite(arr)] = np.nan
        out[b]=arr
    return out

def valid_optical(values):
    optical = np.isfinite(values["S2-L2A-B08"])
    return (optical.sum(1)>=6)&(optical[:,4:10].sum(1)>=3)

def predict_pixels(v,lon,lat,ids):
    labels = np.empty(len(ids),np.uint8)
    scores = np.empty(len(ids),np.float32)
    for start in range(0,len(ids),BATCH):
        ix=ids[start:start+BATCH]
        z=encode({b:a[ix] for b,a in v.items()},lon[ix],lat[ix],DATES,False)
        assert z.shape==(len(ix),6,128) and np.isfinite(z).all()
        p=classifier.predict_proba(z.reshape(len(ix),768))
        assert p.shape==(len(ix),3) and np.isfinite(p).all() and np.allclose(p.sum(1),1)
        labels[start:start+len(ix)]=p.argmax(1)+1
        scores[start:start+len(ix)]=p.max(1)
    return labels,scores

# Smoke test on the first suitable location in the actual cube.
found=False
for y0 in range(0,h,TILE):
    if found: break
    for x0 in range(0,w,TILE):
        tile=ds.isel(y=slice(y0,y0+TILE),x=slice(x0,x0+TILE))
        v=read_values(tile)
        yy,xx=np.meshgrid(tile.y.values,tile.x.values,indexing="ij")
        lon,lat=inverse.transform(xx.ravel(),yy.ravel())
        ids=np.flatnonzero(valid_optical(v)&(maskv[y0:y0+tile.sizes["y"],x0:x0+tile.sizes["x"]].ravel()==1))[:8]
        if len(ids):
            labels,scores=predict_pixels(v,lon,lat,ids)
            print("Pilot labels:",labels,"| raw scores:",scores.round(3))
            found=True
            break
assert found,"No cropland pixels with sufficient optical coverage"


## 7. Tiled spatial inference and GeoTIFF export
Only pixels satisfying cropland and optical coverage criteria receive predictions. Do not store the full 768-feature raster: compute batches directly to keep memory usage low. Codes 1, 2, 3 are the model crop groups, 0 is outside predicted cropland, and 255 is unclassified. The raw score is not calibrated confidence. Neither class counts nor areas represent reliable crop statistics.

In [ ]:
grid=np.full((h,w),255,np.uint8)
grid[maskv==0]=0
confidence=np.full((h,w),-9999,np.float32)
quality=np.zeros((h,w),np.uint8)
for y0 in tqdm(range(0,h,TILE),desc="Spatial inference"):
    for x0 in range(0,w,TILE):
        tile=ds.isel(y=slice(y0,y0+TILE),x=slice(x0,x0+TILE))
        th,tw=tile.sizes["y"],tile.sizes["x"]
        sl=np.s_[y0:y0+th,x0:x0+tw]
        v=read_values(tile)
        ok=valid_optical(v)
        quality[sl]=ok.reshape(th,tw).astype(np.uint8)
        ids=np.flatnonzero(ok&(maskv[sl].ravel()==1))
        if len(ids)==0: continue
        yy,xx=np.meshgrid(tile.y.values,tile.x.values,indexing="ij")
        lon,lat=inverse.transform(xx.ravel(),yy.ravel())
        pred,score=predict_pixels(v,lon,lat,ids)
        rows,cols=np.unravel_index(ids,(th,tw))
        grid[y0+rows,x0+cols]=pred
        confidence[y0+rows,x0+cols]=score
mapped=np.isin(grid,[1,2,3])
assert mapped.any() and np.array_equal(mapped,confidence!=-9999)
assert np.all((confidence[mapped]>=1/3-1e-6)&(confidence[mapped]<=1))
assert np.all(maskv[mapped]==1) and np.all(quality[mapped]==1)
write_raster(RUN/"crop_groups_2022.tif",grid,255,"uint8","Exploratory predicted crop group")
write_raster(RUN/"crop_group_raw_score_2022.tif",confidence,-9999,"float32","Uncalibrated maximum predicted class probability")
write_raster(RUN/"optical_quality_2022.tif",quality,255,"uint8","Optical coverage sufficient (1) or insufficient (0)")
legend = pd.DataFrame({"code":[0,1,2,3,255],"label":["Outside predicted cropland","Cereals","Cotton","Legumes","Unclassified"]})
legend["pixels"]=legend.code.map(lambda k:int((grid==k).sum()))
display(legend)
legend.to_csv(RUN/"crop_group_legend.csv",index=False)
print("Classified fraction of predicted cropland:",mapped.sum()/max(1,(maskv==1).sum()))


## 8. Preview maps and record provenance
The 2022 retrospective evaluation was conducted on reference observations, not on an independently sampled wall-to-wall raster. In the highest score bin, observed accuracy was only 57.5% despite mean confidence 98.2%. Do not interpret the confidence layer as accuracy. The full legend and georeferencing are ready for QGIS.

In [ ]:
colors=["#d9d9d9","#e6a700","#0072b2","#009e73"]
cmap=ListedColormap(colors); cmap.set_bad("white")
norm=BoundaryNorm(np.arange(-.5,4.5),cmap.N)
fig,axes=plt.subplots(1,2,figsize=(12,6),constrained_layout=True)
extent=[transform.c,transform.c+w*10,transform.f-h*10,transform.f]
axes[0].imshow(np.ma.masked_equal(grid,255),cmap=cmap,norm=norm,extent=extent)
axes[0].set_title("(a) Predicted crop groups")
axes[0].legend(handles=[Patch(color=colors[i],label=legend.label.iloc[i]) for i in range(4)],loc="upper center",bbox_to_anchor=(.5,-.04),ncol=2,fontsize=8)
im=axes[1].imshow(np.ma.masked_equal(confidence,-9999),cmap="viridis",vmin=1/3,vmax=1,extent=extent)
axes[1].set_title("(b) Uncalibrated model score")
fig.colorbar(im,ax=axes[1],shrink=.7,label="Maximum predicted probability")
for ax in axes: ax.set_axis_off()
fig.suptitle("Exploratory crop-group mapping | Koumbia, Burkina Faso | 2022")
fig.text(.5,.01,"Retrospective reference evaluation: accuracy 53.05%, macro F1 47.63%; scores are overconfident.",ha="center",fontsize=9)
fig.savefig(RUN/"mapping_preview.png",dpi=200,bbox_inches="tight")
plt.show()
provenance = {"aoi":AOI,"shape":[h,w],"year":MAP_YEAR,"resolution_m":10,
              "checkpoint_sha256":fingerprint,"classifier_sha256":manifest["classifier_sha256"],
              "feature_names":FEATURES,"source":"WorldCereal openEO monthly preprocessed inputs",
              "worldcereal_revision":REVISION,"cropland_mask":"Global annual WorldCereal head, not locally validated",
              "crop_codes":manifest["crop_codes"],"quality_filter":manifest["quality_filter"],
              "confidence":"uncalibrated maximum predicted probability","retrospective_2022":{"accuracy":.5305,"balanced_accuracy":.4936,"macro_f1":.4763},
              "limitations":["Exploratory pilot only","Other crops not represented","Cereals overpredicted","No crop area estimates"]}
(RUN/"mapping_manifest.json").write_text(json.dumps(provenance,indent=2))
print("Output directory:",RUN)
for file in sorted(RUN.glob("*.tif")):
    with rasterio.open(file) as r:
        assert r.shape==(h,w) and r.transform==transform and r.crs.to_epsg()==EPSG
        print(file.name, r.shape,r.nodata,r.crs)
